# Evaluating a Factor: the Information Coefficient

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/lectures/factor_evaluation.ipynb)

Before you ever run a backtest, you can measure whether a signal has *any* predictive power with the
**Information Coefficient (IC)** — the rank correlation between today's signal and tomorrow's return,
across the cross-section. We'll build a real signal and a fake one and learn to tell them apart.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
rng = np.random.default_rng(3)
print("ready")

## 1. A cross-section with a planted edge

`N` stocks over `T` days. Returns are mostly noise, but a **true signal** earns a small predictive
edge (`alpha`). We also include a **noise signal** with no edge — the control.

In [ ]:
N, T, alpha = 100, 500, 0.04
true_signal  = rng.normal(0, 1, (T, N))                 # has predictive power
noise_signal = rng.normal(0, 1, (T, N))                 # has none
# Next-day return = small alpha * today's true signal + lots of noise
returns = alpha * true_signal + rng.normal(0, 1, (T, N))
returns = np.roll(returns, -1, axis=0)                  # align: signal_t predicts return_{t+1}
true_signal, noise_signal, returns = true_signal[:-1], noise_signal[:-1], returns[:-1]
print(f"{N} stocks x {T} days; planted alpha = {alpha}")

## 2. Daily rank IC

For each day, the IC is the **Spearman (rank) correlation** between the signal and next-day returns
across stocks. We use rank correlation so outliers and nonlinearity don't dominate. The series of
daily ICs tells the story: a useful signal has a **positive mean IC** and a high **IC information
ratio** (mean / std of IC).

In [ ]:
def daily_ic(sig, ret):
    return np.array([spearmanr(sig[t], ret[t]).correlation for t in range(len(sig))])

ic_true  = daily_ic(true_signal,  returns)
ic_noise = daily_ic(noise_signal, returns)

def summary(ic, label):
    m, s = np.nanmean(ic), np.nanstd(ic)
    ir = m/s*np.sqrt(252)            # annualized IC information ratio
    hit = np.mean(ic > 0)
    print(f"{label:12} mean IC={m:+.3f}  IC-IR={ir:+.2f}  hit-rate={hit:.0%}")
summary(ic_true,  "true signal")
summary(ic_noise, "noise signal")

The true signal shows a small but **persistently positive** mean IC and a meaningful IC-IR; the noise
signal hovers around zero. A mean IC of even **0.02–0.05** is a genuinely useful equity signal — IC is
small by nature, which is exactly why a single backtest's headline Sharpe is so easy to fool yourself
with.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(np.cumsum(ic_true),  label='true signal',  lw=1.5)
ax.plot(np.cumsum(ic_noise), label='noise signal', lw=1.2, color='grey')
ax.set_title('Cumulative daily IC (slope = predictive power)'); ax.set_xlabel('day'); ax.legend()
plt.tight_layout(); plt.show()

## 3. IC decay: how fast does the edge fade?

Does the signal predict returns 1 day out? 5? 20? Plotting mean IC against the forecast horizon shows
how quickly the edge decays — which tells you how often you must rebalance (and how much turnover/cost
you'll pay).

In [ ]:
def ic_at_horizon(sig, base_returns, h):
    # forward h-day return for each stock, then mean daily rank-IC
    fwd = pd.DataFrame(base_returns).rolling(h).sum().shift(-h).values
    ics = [spearmanr(sig[t], fwd[t]).correlation for t in range(len(sig)-h)]
    return np.nanmean(ics)

# rebuild raw (un-rolled) next-day returns for clean horizon sums
raw = alpha*true_signal + rng.normal(0,1,true_signal.shape)
horizons = [1,2,3,5,10,20]
decay = [ic_at_horizon(true_signal, raw, h)/h for h in horizons]   # per-day IC at each horizon
plt.figure(figsize=(7,3))
plt.plot(horizons, decay, 'o-'); plt.axhline(0,color='grey',lw=0.5)
plt.title('IC decay'); plt.xlabel('forecast horizon (days)'); plt.ylabel('per-day IC'); plt.tight_layout(); plt.show()
print("per-day IC by horizon:", [round(d,3) for d in decay])

## Takeaways

1. **IC = rank correlation of signal vs. next-period return.** Positive, persistent mean IC = a real edge.
2. **Judge the IC information ratio**, not a single day — and remember real equity ICs are *small* (0.02–0.05).
3. **IC decay** sets your horizon and turnover.
4. A noise signal has ~zero IC — the same lesson as p-hacking, viewed cross-sectionally.

Evaluate signals with IC *before* you fall in love with a backtest. Then let the Lab's out-of-sample
grade be the final word. → Try Mission 3 (Alpha Discovery).